# CS UIL Written Test Extractor
Given a json of the question data, generate tex snippets to create

In [2]:
import json
from typing import Literal, get_args
from typing import Any
from os import path, makedirs

Topic = Literal[
    "Simple literal math expression",
    "Simple Output",
    "String class methods",
    "Simple Boolean Logic",
    "Math class methods",
    "Simple variable expression",
    "Conditionals",
    "Simple output loop",
    "1D primitive array",
    "input concepts",
    "accumulation loop",
    "order of operations",
    "java specific data type concepts",
    "ArrayList",
]

TOPIC_LIST: list[str] = list(get_args(Topic))


ContestLevel = Literal["invA", "invB", "District", "Region", "State"]
CONTEST_LEVELS: list[str] = list(get_args(ContestLevel))

print("\n".join(TOPIC_LIST))

Simple literal math expression
Simple Output
String class methods
Simple Boolean Logic
Math class methods
Simple variable expression
Conditionals
Simple output loop
1D primitive array
input concepts
accumulation loop
order of operations
java specific data type concepts
ArrayList


In [3]:
class Question:
    text: str
    code: list[str]
    num: int
    options: list[str]
    competition: "Competition"
    topic: Topic
    correct_idx: int

    def __init__(
        self,
        t: str,
        c: list[str],
        n: int,
        o: list[str],
        comp: "Competition",
        topic: Topic,
        c_idx: int
    ):
        self.text = t
        self.code = c
        self.num = n
        self.options = o
        self.competition = comp
        self.topic = topic
        self.correct_idx = c_idx

    def get_question_id(self) -> str:
        return f"{self.competition.year}{self.competition.get_letter()}{self.num}"

    def to_latex(self) -> str:
        if self.code is not None and len(self.code) > 0:
            return self.__code_latex__()
        return self.__latex__()
    
    def __create_choices__(self):        
        choices = []
        for i, o in enumerate(self.options):
            question_text = ""
            if i == self.correct_idx:
                question_text = f"\\CorrectChoice{{{o}}}"
            else:
                question_text = f"\\choice {o}"
            choices.append(question_text)
        return choices

    def __latex__(self):
        choices = self.__create_choices__()
        return f"""
\\begin{{questionc}}
    \\begin{{qtext}}{{{self.get_question_id()}}}
{self.text}
    \\end{{qtext}}
    \\begin{{qchoices}}
        {"\n".join(choices)}
    \\end{{qchoices}}
\\end{{questionc}}
"""

    def __code_latex__(self):
        choices = self.__create_choices__()
        return f"""
\\begin{{questionc}}
    \\begin{{qtext}}{{{self.get_question_id()}}}
{self.text}
    \\end{{qtext}}
    \\begin{{minted}}{{java}}
{"\n".join(self.code)}
    \\end{{minted}}
    \\begin{{qchoices}}
        {"\n".join(choices)}
    \\end{{qchoices}}
\\end{{questionc}}
"""

    def __str__(self) -> str:
        return f"[{self.num}] {self.text}\n\t{"\t".join(self.options)}"

    def get_path_strand(self) -> str:
        return f'{self.topic.lower().replace(" ", "_")}/'


class Competition:
    level: ContestLevel
    year: int
    questions: list[Question]

    def __init__(self, level: ContestLevel, year: int):
        self.level = level
        self.year = year
        self.questions = []

    def __str__(self) -> str:
        representation = f"{self.year}{self.level}"
        for q in self.questions:
            representation += "\n\t" + str(q)
        return representation

    def add_question(self, q: Question):
        self.questions.append(q)

    def get_letter(self):
        if self.level == "invA":
            return "A"
        elif self.level == "invB":
            return "B"
        elif self.level == "District":
            return "D"
        elif self.level == "Region":
            return "R"
        elif self.level == "State":
            return "S"

    def from_file(year: int, level: ContestLevel, file_path: str) -> "Competition":
        C = Competition(level, year)
        values = None
        with open(file_path) as file:
            values = json.load(file)
        if values is None:
            raise Exception("failure!")

        # print(json.dumps(values, indent=2))
        for v in values:
            qn = v["question_number"]
            text = v["question_text"]
            code = str(v["code_snippet"])
            correct_text = v["correct"]
            options_raw = v["options"]
            correct_idx = -1
            options = []
            for i, op in enumerate(options_raw):
                op_text = op.replace("%^%", "\\\\").replace("[", "\\[").replace("]", "\\]")
                options.append(op_text)
                if op == correct_text:
                    correct_idx = i

            code_parts = code.split("%^%")
            topic = v["topic"]

            # print(f"Read the following data:\n\ttext{{{text}}}\n\t{{{code_parts}}}\n\t{{{"\n".join(options)}}}")
            Q = Question(text, code_parts, qn, options, C, topic, correct_idx)
            if topic != Q.topic:
                print(f"MISMATCH: {qn} {text}")
            C.add_question(Q)

        return C

    def write_latex(self, base_output_path="./out/"):
        num_written = 0
        for q in self.questions:
            output_dir = path.join(base_output_path, q.get_path_strand())
            makedirs(output_dir, exist_ok=True)
            output_path = path.join(output_dir, f"{self.year}{self.level}.tex")
            with open(output_path, "w") as file:
                file.write(q.to_latex())
            num_written += 1
            # print(f"wrote topic {q.topic.lower().replace(" ", "_")}")
        # print(f"wrote {num_written} files")

    # def create_test:

In [4]:
def collect_competitions(year, debug=False):
    C = []
    for level in get_args(ContestLevel):
        fp = f"./data/{year}/{level.lower()}.json"
        if path.exists(fp) is False:
            print(f"Unable to find [{fp}] !")
            continue
        print(f"Reading data from {year} {level}...")
        c = Competition.from_file(year, level, fp)
        if debug:
            print(f"{c.year} {c.level}")
            for q in c.questions:
                print(q)
        C.append(c)
    return C

def generate_latex(year: int, debug=False):
    C = collect_competitions(year, debug)
    for c in C:
        c.write_latex()
    # for level in get_args(ContestLevel):
    #     fp = f"./data/{year}/{level.lower()}.json"
    #     if path.exists(fp) is False:
    #         print(f"Unable to find [{fp}] !")
    #         continue
    #     c = Competition.from_file(year, level, fp)
    #     # ipdb.set_trace()
    #     c.write_latex()
    #     print(f"Wrote data from {year} {level}")


In [5]:
f = lambda t: t.lower().replace(" ", "_")
topic_normalized = map(f, TOPIC_LIST)
# for t in topic_normalized:
#     print(f"\\def\\topic{{{t}}}")


print("\n\n")
print(f'\\def\\topics{{{", ".join(topic_normalized)}}}')





\def\topics{simple_literal_math_expression, simple_output, string_class_methods, simple_boolean_logic, math_class_methods, simple_variable_expression, conditionals, simple_output_loop, 1d_primitive_array, input_concepts, accumulation_loop, order_of_operations, java_specific_data_type_concepts, arraylist}


In [6]:
C = collect_competitions(2025)

Reading data from 2025 invA...
Reading data from 2025 invB...
Reading data from 2025 District...
Reading data from 2025 Region...
Reading data from 2025 State...


In [7]:
years = [2025, 2024, 2023, 2022]
for year in years:
    generate_latex(year, debug=False)
    print("="*30)

Reading data from 2025 invA...
Reading data from 2025 invB...
Reading data from 2025 District...
Reading data from 2025 Region...
Reading data from 2025 State...
Reading data from 2024 invA...
Reading data from 2024 invB...
Reading data from 2024 District...
Reading data from 2024 Region...
Reading data from 2024 State...
Reading data from 2023 invA...
Reading data from 2023 invB...
Reading data from 2023 District...
Reading data from 2023 Region...
Reading data from 2023 State...
Reading data from 2022 invA...
Reading data from 2022 invB...
Reading data from 2022 District...
Reading data from 2022 Region...
Reading data from 2022 State...
